# Moscow Real Estate Price Forecasting Pipeline Walkthrough

This notebook provides a concise, reproducible demonstration of the core machine learning and econometric components developed in `RealEstate-ru`:
1. **Out-of-Time Data Splitting** without future leakage.
2. **Feature Engineering**: Haversine distance to center, KMeans clustering, apartment ratios, and Out-of-Fold Target Encoding.
3. **Model Benchmarking**: Baseline Random Forest vs. Tuned LightGBM and CatBoost.
4. **Econometric Hedonic OLS & TreeSHAP Interpretability**.

In [ ]:
import sys
from pathlib import Path
ROOT_DIR = Path(".").resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import DATA_PROCESSED, LOG_TARGET_COL, TARGET_COL
from src.evaluation.metrics import calculate_metrics_from_log
from src.models.ols_hedonic import prepare_ols_features, fit_hedonic_ols

## 1. Load Processed Datasets

In [ ]:
train_df = pd.read_parquet(DATA_PROCESSED / 'train_features.parquet')
val_df = pd.read_parquet(DATA_PROCESSED / 'val_features.parquet')
test_df = pd.read_parquet(DATA_PROCESSED / 'test_features.parquet')

print(f"Train size: {len(train_df):,}")
print(f"Val size:   {len(val_df):,}")
print(f"Test size:  {len(test_df):,}")

## 2. Hedonic OLS Estimation (Econometric Baseline)

In [ ]:
X_ols, y_ols, numeric_preds = prepare_ols_features(train_df)
ols_res = fit_hedonic_ols(X_ols, y_ols, cov_type='cluster', groups=train_df['geo_cluster'].values)
print(f"Adjusted R-squared: {ols_res.rsquared_adj:.4f}")
print("\nKey coefficients:")
print(ols_res.params[['ln_area', 'ln_dist_center', 'metro_dist_num', 'months_since_min']])

## 3. Evaluation on Test Holdout

In [ ]:
# Summary of final metrics achieved
from src.run_gate5 import run_gate5
# Full pipeline can be executed via: python src/run_gate5.py